# Xuất field bổ sung cho dashboard cá nhân hoá

Notebook 01 (`01_personal_summary_3tier.ipynb`) và 02 (`02_session_timeseries_export.ipynb`)
đã xuất phần khung chính cho dashboard cá nhân hoá (Tier 1/2/3,
session timeseries). Trong lúc chỉnh dashboard theo yêu cầu (thêm biểu đồ LF/HF,
Sóng thư giãn alpha, Chỉ số gắn kết, Điểm cảm xúc theo sóng não...), một số field
đã được merge **trực tiếp bằng script Python ad-hoc** vào file HTML, KHÔNG qua
notebook nào — nghĩa là nếu dữ liệu gốc cập nhật (thêm session mới) và chạy lại
pipeline từ đầu, các field này sẽ không tự tái tạo.

Notebook này codify lại toàn bộ các field đó thành 1 bước xuất chính thức,
tái sử dụng đúng phương pháp đã dùng lúc merge thủ công (đã verify khớp với
kết quả gốc trước khi đưa vào dashboard).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

REPO_ROOT = Path("../..").resolve()
META_PATH = REPO_ROOT / "data" / "analytics" / "metadata_multimodal" / "session_metadata_survey_features.csv"
COMPOSITE_PATH = REPO_ROOT / "data" / "analytics" / "composite_eeg_mood_score_per_session.csv"
OUT_DIR = Path(".").resolve() / "outputs"
OUT_DIR.mkdir(exist_ok=True)

metadata = pd.read_csv(META_PATH, low_memory=False)
print("metadata:", metadata.shape)

out = metadata[["participant_key", "study_day"]].copy()

## 1. Δ EEG theo ngày (tab 1 — mục "Xem chỉ số chi tiết")

Đổi tên cột `delta_eeg_mean_*` / `delta_rel_eeg_mean_*` sang tên field dashboard
đang dùng. `delta_engagement_index` = Δ theta/(alpha+beta), lấy từ cột
`delta_eeg_mean_theta_alpha_beta_ratio` đã có sẵn trong file gốc.

**Đính chính**: tên cột `engagement_index`/`engage_*` là tên gọi
SAI, giữ lại vì lý do tương thích ngược (đổi tên cột sẽ phải sửa nhiều nơi
khác đang tham chiếu). Theo đúng `eeg_feature_extraction.ipynb` (trích Jap et
al. 2009), theta/(alpha+beta) là chỉ số mệt mỏi tinh thần/buồn ngủ, KHÔNG
phải Engagement Index thật — Engagement Index thật (Pope et al. 1995) là công
thức khác hẳn: beta/(theta+alpha) (cột `eeg_cross_FEI`, hiện không dùng trên
UI). Nhãn hiển thị cho người dùng trên dashboard đã sửa lại đúng thành "Chỉ số mệt mỏi tinh thần".

In [ ]:
out["delta_theta_power_rel"] = metadata["delta_eeg_mean_theta_power_rel"].round(4)
out["delta_theta_alpha_ratio"] = metadata["delta_eeg_mean_theta_alpha_ratio"].round(3)
out["delta_beta_power_rel"] = metadata["delta_eeg_mean_beta_power_rel"].round(4)
out["delta_engagement_index"] = metadata["delta_eeg_mean_theta_alpha_beta_ratio"].round(3)

print(out[["delta_theta_power_rel", "delta_theta_alpha_ratio", "delta_beta_power_rel", "delta_engagement_index"]].notna().sum())

## 2. LF/HF ratio theo pha (tab 1 & tab 2 — panel "trước → trong → sau thiền")

Cột `ppg_lf_hf_ratio__{PHASE}` đã có sẵn trong file gốc (tính từ HRV miền
tần số), chỉ cần lấy ra và làm tròn.

In [ ]:
PHASES = ["REST_BEFORE", "MEDITATION", "REST_AFTER"]

for ph in PHASES:
    out[f"ppg_lf_hf_ratio__{ph}"] = metadata[f"ppg_lf_hf_ratio__{ph}"].round(3)

print(out[[f"ppg_lf_hf_ratio__{ph}" for ph in PHASES]].notna().sum())

## 3. Alpha power & Chỉ số mệt mỏi tinh thần theo pha (tab 2 — panel "Sóng não")

Giá trị THÔ (không phải delta) của `eeg_mean_alpha_power_rel` và
`eeg_mean_theta_alpha_beta_ratio` tại từng pha — dùng để vẽ 3 điểm
trước/trong/sau nối bằng đường, giống panel HR/RMSSD/LF-HF. Tên cột nội bộ
vẫn là `engage_*` (xem đính chính ở mục 1) — chỉ đổi tên hiển thị.

In [ ]:
for ph in PHASES:
    out[f"alpha_{ph}"] = metadata[f"eeg_mean_alpha_power_rel__{ph}"].round(4)
    out[f"engage_{ph}"] = metadata[f"eeg_mean_theta_alpha_beta_ratio__{ph}"].round(4)

print(out[[f"alpha_{ph}" for ph in PHASES] + [f"engage_{ph}" for ph in PHASES]].notna().sum())

## 4. Composite EEG-mood score theo pha (tab 2 — panel "Sóng não")

`01_composite_eeg_mood_score.ipynb` chỉ xuất bản **delta** (`composite_abs_MEDITATION`
= z-score(MEDITATION) − z-score(REST_BEFORE), đã canh dấu theo tương quan với
Δmood). Panel "Sóng não: trước → trong → sau thiền" cần điểm composite **riêng
cho từng pha** để vẽ 3 điểm nối đường — không có sẵn, phải tính lại.

**Phương pháp**: tái sử dụng đúng 7 feature + cách xác định dấu (`SIGN`) của
notebook 01, nhưng thay vì z-score trên phân bố của **delta**, z-score trên
phân bố **giá trị thô gộp cả 3 pha** (pooled) — để 3 điểm trước/trong/sau nằm
trên cùng 1 thang đo, so sánh được với nhau.

In [ ]:
CLUSTER_FEATURES = [
    "mean_hjorth_mobility", "mean_hjorth_complexity", "mean_renyi_entropy",
    "mean_beta_power_rel", "mean_gamma_power_rel", "mean_high_beta_power_rel",
    "mean_theta_alpha_beta_ratio",
]

# 4a. Xac dinh dau tung feature -- giong het notebook 01 (dua tren delta REST_AFTER-REST_BEFORE vs delta_mood)
for f in CLUSTER_FEATURES:
    a, b = f"eeg_{f}__REST_BEFORE", f"eeg_{f}__REST_AFTER"
    metadata[f"delta_rest_eeg_{f}"] = metadata[b] - metadata[a]

SIGN = {}
for f in CLUSTER_FEATURES:
    col = f"delta_rest_eeg_{f}"
    sub = metadata[[col, "delta_mood"]].dropna()
    r, p = stats.spearmanr(sub[col], sub["delta_mood"])
    SIGN[f] = 1 if r >= 0 else -1

print("SIGN (khop notebook 01):", SIGN)

In [ ]:
# 4b. Z-score tren phan bo GOP CA 3 PHA cho tung feature, roi ap dau + trung binh 7 feature (>=4/7 hop le)
zscored = {}
for f in CLUSTER_FEATURES:
    cols = [f"eeg_{f}__{ph}" for ph in PHASES]
    pooled = pd.concat([metadata[c] for c in cols], axis=0)
    mu, sd = pooled.mean(), pooled.std(ddof=0)
    for ph in PHASES:
        zscored[(f, ph)] = SIGN[f] * (metadata[f"eeg_{f}__{ph}"] - mu) / sd

for ph in PHASES:
    z = pd.DataFrame({f: zscored[(f, ph)] for f in CLUSTER_FEATURES})
    n_valid = z.notna().sum(axis=1)
    out[f"composite_eeg_mood__{ph}"] = z.mean(axis=1, skipna=True).where(n_valid >= 4).round(3)

print(out[[f"composite_eeg_mood__{ph}" for ph in PHASES]].describe())

### Sanity check

So sánh `composite_eeg_mood__MEDITATION − composite_eeg_mood__REST_BEFORE`
(tính theo phương pháp mới, pooled-phase z-score) với `composite_abs_MEDITATION`
chính thức từ notebook 01 (z-score trên phân bố delta) — 2 cách tính khác nhau
về mặt cơ sở z-score, nên kỳ vọng tương quan rất cao nhưng không nhất thiết
tuyệt đối = 1.0.

In [ ]:
composite_official = pd.read_csv(COMPOSITE_PATH)
check = out.merge(composite_official, on=["participant_key", "study_day"], how="inner")
diff = check["composite_eeg_mood__MEDITATION"] - check["composite_eeg_mood__REST_BEFORE"]
valid = diff.notna() & check["composite_abs_MEDITATION"].notna()
r, p = stats.spearmanr(diff[valid], check.loc[valid, "composite_abs_MEDITATION"])
print(f"So sanh voi composite_abs_MEDITATION chinh thuc: r={r:.4f}, p={p:.2e}, n={valid.sum()}")
assert r > 0.99, "Lech qua nhieu so voi composite chinh thuc -- kiem tra lai phuong phap"
print("OK -- khop voi composite chinh thuc, an toan de dung cho dashboard.")

## 5. Điểm cảm xúc theo sóng não dạng delta (tab 1 — trước đã dùng, có thể tái dùng sau này)

Giữ luôn bản delta chính thức (`composite_abs_MEDITATION`) dưới tên field
`composite_eeg_mood` mà dashboard đã dùng, để notebook này là nguồn đầy đủ
duy nhất cho mọi field composite EEG-mood.

In [ ]:
out = out.merge(
    composite_official[["participant_key", "study_day", "composite_abs_MEDITATION"]]
        .rename(columns={"composite_abs_MEDITATION": "composite_eeg_mood"}),
    on=["participant_key", "study_day"], how="left",
)
out["composite_eeg_mood"] = out["composite_eeg_mood"].round(3)
print(out["composite_eeg_mood"].notna().sum(), "/", len(out))

## 6. Xuất file

In [ ]:
export_path = OUT_DIR / "dashboard_extra_fields.csv"
out.to_csv(export_path, index=False)
print("Da luu:", export_path, "--", out.shape)
out.head()